# Code Execution and Sandboxing

Notebook **07** covers **code execution** in AutoGen 0.4+: **`CodeExecutorAgent`**, **`LocalCommandLineCodeExecutor`** from **`autogen_ext`**, a **Docker executor sketch**, **approval functions**, and the **`UserProxyAgent`** pattern from **04**. We generate a **pytest snippet** for the Notes API but execute it through a **mock executor** — never arbitrary dangerous code in teaching notebooks.

**Prerequisites:** **04. UserProxyAgent and Human Input**, **06. Tools and Function Registration**.

**What you'll learn:**

- When to use **`CodeExecutorAgent`** vs **`PythonCodeExecutionTool`**
- How **`approval_func`** gates code before execution
- Sandbox limits, guardrails, and when **not** to run LLM-generated code
- How to sketch Docker isolation without requiring Docker in the notebook

In [ ]:
OPENAI_API_KEY = "sk-proj-placeholder-replace-with-your-real-key"

try:
    import autogen_agentchat
    print("autogen-agentchat:", getattr(autogen_agentchat, "__version__", "installed"))
except ImportError:
    print("autogen-agentchat: pip install autogen-agentchat autogen-ext[openai]")

try:
    import autogen_ext
    print("autogen-ext:", getattr(autogen_ext, "__version__", "installed"))
except ImportError:
    print("autogen-ext: pip install autogen-ext[openai]")

---

## 1. Why Code Execution?

Some tasks need **running** code — plots, data transforms, test scaffolding. The model writes markdown code blocks; an executor runs them and returns stdout/stderr.

| Pattern | Use when |
|---------|----------|
| **Tools only (06)** | Structured lookups — no free-form code |
| **`PythonCodeExecutionTool`** | AssistantAgent tool with bounded execution |
| **`CodeExecutorAgent`** | Dedicated executor in a multi-agent team |
| **Never execute** | Untrusted input, production secrets, unknown packages |

**Rule:** treat all LLM-generated code as **untrusted** until reviewed.

---

## 2. Execution Architecture

```
  AssistantAgent                CodeExecutorAgent
  (writes ```python blocks)  →  (extracts & executes)
         │                              │
         │                              ▼
         │                    ┌──────────────────┐
         └──── team chat ────►│  CodeExecutor    │
                              │  local / docker  │
                              └──────────────────┘
```

**`CodeExecutorAgent`** can run **with** or **without** its own model — in teams it often executes blocks from other agents' messages.

---

## 3. Notes API Context

We'll ask agents to generate a **pytest** snippet for Notes API tests — grounded in corpus chunk **c4** (fixtures in **`conftest.py`**).

In [ ]:
NOTES = {
    "n1": "The Notes API is built with FastAPI. Routes live under /notes.",
    "n2": "Run alembic upgrade head after pulling database migrations.",
    "n3": "JWT bearer tokens use Authorization: Bearer header.",
    "n4": "Pytest fixtures for DB setup belong in conftest.py.",
}

DOC_CHUNKS = [
    {"id": "c1", "text": NOTES["n1"]},
    {"id": "c2", "text": NOTES["n2"]},
    {"id": "c3", "text": NOTES["n3"]},
    {"id": "c4", "text": NOTES["n4"]},
    {"id": "c5", "text": "Use httpx.AsyncClient in FastAPI tests for async endpoints."},
]

print("corpus:", [c["id"] for c in DOC_CHUNKS])

---

## 4. Mock Executor — Safe Teaching Default

For notebooks and unit tests, use a **mock executor** that records code but does not run it on the host:

In [ ]:
from typing import List

from autogen_core import CancellationToken
from autogen_core.code_executor import CodeBlock, CodeExecutor, CodeResult


class MockCodeExecutor(CodeExecutor):
    """Records code blocks without executing them — safe for teaching."""

    def __init__(self) -> None:
        self.executed_blocks: List[CodeBlock] = []

    async def execute_code_blocks(
        self, code_blocks: List[CodeBlock], cancellation_token: CancellationToken
    ) -> CodeResult:
        self.executed_blocks.extend(code_blocks)
        combined = "\n---\n".join(f"[{b.language}]\n{b.code}" for b in code_blocks)
        return CodeResult(
            exit_code=0,
            output=f"MOCK EXECUTION (not run on host):\n{combined}",
        )

    async def start(self) -> None:
        return None

    async def stop(self) -> None:
        return None

    async def restart(self) -> None:
        self.executed_blocks.clear()


mock_executor = MockCodeExecutor()
print("mock executor ready")

---

## 5. `CodeExecutorAgent` with Mock Executor

**`CodeExecutorAgent`** scans messages for markdown code fences and passes blocks to the executor:

In [ ]:
from autogen_agentchat.agents import CodeExecutorAgent
from autogen_agentchat.messages import TextMessage
from autogen_core import CancellationToken

executor_agent = CodeExecutorAgent(
    name="code_runner",
    code_executor=mock_executor,
    description="Executes python code blocks from team messages (mocked here).",
)

sample_message = TextMessage(
    source="assistant",
    content="""Here is a pytest stub:

```python
# test_notes.py — generated for teaching
def test_list_notes(client):
    response = client.get("/notes")
    assert response.status_code == 200
```
""",
)

exec_result = await executor_agent.on_messages([sample_message], CancellationToken())
print(exec_result.chat_message.content[:400])

---

## 6. `LocalCommandLineCodeExecutor` — Commented Safe Demo

**`LocalCommandLineCodeExecutor`** from **`autogen_ext.code_executors.local`** writes each block to **`work_dir`** and runs it in a subprocess. Keep this cell commented unless you trust the code:

In [ ]:
# from pathlib import Path
# from autogen_core import CancellationToken
# from autogen_core.code_executor import CodeBlock
# from autogen_ext.code_executors.local import LocalCommandLineCodeExecutor
#
# work_dir = Path("coding_workspace")
# work_dir.mkdir(exist_ok=True)
# local_executor = LocalCommandLineCodeExecutor(work_dir=work_dir, timeout=30)
#
# trusted = await local_executor.execute_code_blocks(
#     [CodeBlock(language="python", code="print('trusted hello')")],
#     CancellationToken(),
# )
# print(trusted.output)

print("LocalCommandLineCodeExecutor demo is commented out for safety.")

---

## 7. Docker Executor Sketch

**`DockerCommandLineCodeExecutor`** (**`autogen-ext[docker]`**) isolates execution in a container:

```python
# Requires Docker daemon + pip install autogen-ext[docker]
# from autogen_ext.code_executors.docker import DockerCommandLineCodeExecutor
#
# docker_executor = DockerCommandLineCodeExecutor(work_dir="coding")
# await docker_executor.start()
# try:
#     result = await docker_executor.execute_code_blocks(blocks, CancellationToken())
# finally:
#     await docker_executor.stop()
```

Use Docker when the model generates install commands, file writes, or network calls.

---

## 8. `CodeBlock` Extraction Pattern

Executors consume **`CodeBlock`** objects — language tag + code string. Fences in markdown map to blocks:

In [ ]:
from autogen_core.code_executor import CodeBlock

pytest_block = CodeBlock(
    language="python",
    code="""def test_health(client):
    assert client.get("/health").status_code == 200
""",
)

mock_result = await mock_executor.execute_code_blocks(
    [pytest_block],
    CancellationToken(),
)
print(mock_result.output[:300])

---

## 9. Approval Function Pattern

**`approval_func`** on **`CodeExecutorAgent`** runs **before** each execution. Return **`ApprovalResponse(approved=False)`** to block dangerous patterns:

In [ ]:
from autogen_agentchat.agents import ApprovalRequest, ApprovalResponse

BLOCKED_PATTERNS = ["os.system", "subprocess", "rm -rf", "shutil.rmtree", "__import__"]


def guardrail_approval(request: ApprovalRequest) -> ApprovalResponse:
    code = request.code or ""
    for pattern in BLOCKED_PATTERNS:
        if pattern in code:
            return ApprovalResponse(
                approved=False,
                reason=f"Blocked pattern detected: {pattern}",
            )
    return ApprovalResponse(approved=True, reason="Passed static guardrails")


sample_code = "import os\nos.system('rm -rf /')"
req = ApprovalRequest(code=sample_code, context=[])
decision = guardrail_approval(req)
print("approved:", decision.approved, "|", decision.reason)

---

## 10. Approval Function on `CodeExecutorAgent`

Wire **`guardrail_approval`** into the agent so blocked patterns never reach the executor:

In [ ]:
guarded_runner = CodeExecutorAgent(
    name="guarded_runner",
    code_executor=mock_executor,
    approval_func=guardrail_approval,
)

unsafe = TextMessage(
    source="assistant",
    content="""```python
import os
os.system('echo unsafe')
```""",
)

guarded_out = await guarded_runner.on_messages([unsafe], CancellationToken())
print(guarded_out.chat_message.content[:300])

---

## 11. `UserProxyAgent` Code Approval (**04**)

For **human** approval, pair **`UserProxyAgent`** with a coding assistant. In production, wire **`input_func`** to a web socket — not blocking **`input()`** (**04**).

In [ ]:
from autogen_agentchat.agents import UserProxyAgent

def auto_approve_input(prompt: str = "") -> str:
    """Mock user — auto-approves safe teaching stubs only."""
    return "APPROVE — safe pytest stub reviewed."


user_proxy = UserProxyAgent(name="user_proxy", input_func=auto_approve_input)
print("user_proxy ready:", user_proxy.name)

---

## 12. Sandbox Limits

| Limit | Purpose |
|-------|---------|
| **`timeout`** | Kill runaway loops (default ~60s local) |
| **`work_dir`** | Contain file writes to one directory |
| **Virtual env context** | Isolate pip installs from host Python |
| **Language allowlist** | **`supported_languages`** on CodeExecutorAgent |
| **Static guardrails** | Block dangerous imports before execution |
| **Mock executor** | Zero host risk for demos and CI |

Combine limits — no single layer is sufficient.

---

## 13. When NOT to Run Arbitrary Code

**Do not execute** LLM-generated code when:

- Input includes **user-provided** code mixed with model output
- The environment has **secrets** (API keys, DB URLs) in env vars
- The host is **shared** (notebook server, CI without containers)
- The task only needs **text** answers — use tools (**06**) instead
- Packages would be **pip installed** from unverified indexes

Generate → **human review** → execute in **Docker** is the production pattern.

---

## 14. Demo — Generate Pytest Snippet (Mock Execution)

Use an **`AssistantAgent`** to draft a pytest stub, then pass the message to the mock **`CodeExecutorAgent`**:

In [ ]:
from autogen_ext.models.openai import OpenAIChatCompletionClient
from autogen_agentchat.agents import AssistantAgent

model_client = OpenAIChatCompletionClient(
    model="gpt-4o-mini",
    api_key=OPENAI_API_KEY,
    temperature=0,
)

test_writer = AssistantAgent(
    name="test_writer",
    model_client=model_client,
    system_message=(
        "You write minimal pytest stubs for the FastAPI Notes API. "
        "Put code in a single ```python fenced block. "
        "Reference conftest.py fixtures per doc chunk c4. "
        "Do not use os, subprocess, or network calls."
    ),
)

draft = await test_writer.run(
    task="Write a pytest function test_get_notes_returns_200 using a client fixture."
)
draft_text = draft.messages[-1].content
print(draft_text[:500])

await mock_executor.restart()
mock_runner = CodeExecutorAgent(name="mock_runner", code_executor=mock_executor)
run_out = await mock_runner.on_messages(
    [TextMessage(source="test_writer", content=draft_text)],
    CancellationToken(),
)
print("\n--- executor output ---")
print(run_out.chat_message.content[:500])
print("\nblocks captured:", len(mock_executor.executed_blocks))

---

## 15. `PythonCodeExecutionTool` Alternative

Microsoft recommends **`AssistantAgent` + `PythonCodeExecutionTool`** as an alternative to **`CodeExecutorAgent`** when the model should call execution as a **tool** (**06**):

```python
from autogen_ext.tools.code_execution import PythonCodeExecutionTool
from autogen_ext.code_executors.local import LocalCommandLineCodeExecutor

# tool = PythonCodeExecutionTool(MockCodeExecutor())  # use mock in teaching
# agent = AssistantAgent("assistant", model_client, tools=[tool], reflect_on_tool_use=True)
```

The model must emit properly escaped code strings as tool parameters — stricter but fits single-agent flows.

---

## 16. Team Layout — Writer + Executor

In multi-agent teams, a **coder** agent writes blocks; **`CodeExecutorAgent`** executes them (**08**):

```
RoundRobinGroupChat([test_writer, code_runner], ...)
```

Set **`sources=["test_writer"]`** on **`CodeExecutorAgent`** to ignore code from other agents.

In [ ]:
from autogen_agentchat.teams import RoundRobinGroupChat
from autogen_agentchat.conditions import MaxMessageTermination

await mock_executor.restart()
scoped_runner = CodeExecutorAgent(
    name="scoped_runner",
    code_executor=mock_executor,
    sources=["test_writer"],
)

# Teaching team — executor only processes test_writer code blocks
code_team = RoundRobinGroupChat(
    participants=[test_writer, scoped_runner],
    termination_condition=MaxMessageTermination(max_messages=4),
    max_turns=4,
)

# team_result = await code_team.run(task="Write and 'run' a tiny pytest assert True stub.")
print("code_team participants:", [a.name for a in code_team._participants])

---

## 17. Design Guidelines

| Guideline | Rationale |
|-----------|-----------|
| **Default to mock/Docker** | Host execution is dangerous with LLM code |
| **`approval_func` for automation** | Static checks before human review |
| **`UserProxyAgent` for short gates only** | Blocking breaks save/resume (**04**) |
| **`sources` filter** | Only execute trusted agent messages |
| **Timeouts + work_dir** | Limit blast radius on local executor |
| **Prefer tools over code** | Structured tools are safer than free-form Python |

---

## 18. Common Mistakes

| Mistake | Consequence | Fix |
|---------|-------------|-----|
| Running local executor on LLM code | Host compromise | Docker or mock |
| No timeout | Infinite loops burn CPU | Set **`timeout=30`** |
| Auto-approving everything | Silent malicious runs | **`approval_func`** + human gate |
| Long `UserProxyAgent` waits | Unstable team state | External approval API |
| Executing all agents' code | Wrong block executed | **`sources=[...]`** filter |
| pip install in notebook host | Dependency pollution | Container + venv context |

---

## 19. Summary

Code execution is powerful and risky. **`CodeExecutorAgent`** extracts markdown blocks and delegates to a **`CodeExecutor`** — local, Docker, or **mock**. Use **`approval_func`** and **`UserProxyAgent`** (**04**) for gates; prefer **mock executors** in teaching and CI.

Key takeaways:

- **`LocalCommandLineCodeExecutor`** and Docker are for **trusted, reviewed** code only.
- Generate pytest stubs for the Notes API, but **mock-execute** in this notebook.
- **`PythonCodeExecutionTool`** integrates execution as a tool on **`AssistantAgent`** (**06**).

Demonstrations built a mock executor, ran **`CodeExecutorAgent`** on a pytest snippet, applied static guardrails, and sketched local/Docker paths.

Next: **08. GroupChat and Multi-Agent Teams** — three specialist agents collaborate on multi-part Notes API questions.